# EV3 Notebook 3: Ingenieria de Caracteristicas
## Bank Marketing Dataset

Creacion de nuevas variables derivadas que capturen relaciones no lineales y enriquezcan el dataset para mejorar el rendimiento de los modelos predictivos.

**Anterior:** `EV3_02_Escalamiento_Codificacion.ipynb`  
**Siguiente:** `EV3_04_Machine_Learning.ipynb`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
os.makedirs('graficos', exist_ok=True)

# Cargar dataset
df = pd.read_csv('bank-additional-full.csv', sep=';')
df = df.drop_duplicates()

# Descartar 'duration' por ser data leaker
if 'duration' in df.columns:
    df = df.drop(columns=['duration'])

# Renombrar columnas al espanol
rename_columns = {
    'y': 'deposito_plazo', 'age': 'edad', 'job': 'trabajo',
    'marital': 'estado_civil', 'education': 'educacion', 'default': 'mora',
    'housing': 'vivienda', 'loan': 'prestamo', 'contact': 'contacto',
    'month': 'mes', 'day_of_week': 'dia_de_la_semana', 'campaign': 'campana',
    'pdays': 'dias_previos', 'previous': 'anterior', 'poutcome': 'resultado_anterior',
    'emp.var.rate': 'var_empleo', 'cons.price.idx': 'indice_precios',
    'cons.conf.idx': 'indice_confianza', 'euribor3m': 'tasa_euribor',
    'nr.employed': 'num_empleados'
}
df = df.rename(columns=rename_columns)
df['deposito_plazo'] = df['deposito_plazo'].map({'yes': 'si', 'no': 'no'})
df['deposito_plazo_num'] = df['deposito_plazo'].map({'si': 1, 'no': 0})
df_ml = df.copy()

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")

In [ ]:
# Seccion 1 aplicada: imputacion de unknowns por moda
cols_unknown = ['trabajo', 'estado_civil', 'educacion', 'mora', 'vivienda', 'prestamo']
for col in cols_unknown:
    df_ml[col] = df_ml[col].replace('unknown', df_ml[df_ml[col] != 'unknown'][col].mode()[0])
print("Imputacion aplicada.")

In [ ]:
# Seccion 2 aplicada: capping IQR y StandardScaler
from sklearn.preprocessing import StandardScaler

def cap_iqr(s):
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    return s.clip(lower=Q1 - 1.5*(Q3-Q1), upper=Q3 + 1.5*(Q3-Q1))

df_ml['edad'] = cap_iqr(df_ml['edad'])
df_ml['campana'] = cap_iqr(df_ml['campana'])

numeric_cols_ml = ['edad', 'campana', 'dias_previos', 'anterior',
                   'var_empleo', 'indice_precios', 'indice_confianza',
                   'tasa_euribor', 'num_empleados']
scaler = StandardScaler()
df_ml[numeric_cols_ml] = scaler.fit_transform(df_ml[numeric_cols_ml])
print("Escalamiento aplicado.")

In [ ]:
# Seccion 3 aplicada: One-Hot Encoding
cat_cols_ml = df_ml.select_dtypes(include=['object']).columns.drop('deposito_plazo').tolist()
df_ml = pd.get_dummies(df_ml, columns=cat_cols_ml, drop_first=True)
bool_cols = df_ml.select_dtypes(include=['bool']).columns
df_ml[bool_cols] = df_ml[bool_cols].astype(int)
print("OHE aplicado.")

## Seccion 4: Ingenieria de Caracteristicas

Se crean nuevas variables a partir de las existentes para capturar relaciones no lineales que los algoritmos podrian no detectar automaticamente.

Las variables se crean sobre `df` (dataset original sin escalar) para mantener interpretabilidad en las visualizaciones.

| Variable nueva | Formula / Logica | Justificacion |
|---|---|---|
| es_jubilado | 1 si trabajo == jubilado | Jubilados mostraron la mayor tasa de conversion en el EDA |
| contactado_antes | 1 si dias_previos != 999 | Tener historial previo es el segundo predictor mas importante |
| contexto_favorable | 1 si euribor < 2 y var_empleo < 0 | Tasas bajas hacen atractivos los depositos a plazo |
| intensidad_campana | campana / (anterior + 1) | Ratio de insistencia nueva vs historial previo |
| grupo_edad | bins: Joven <= 30, Adulto 31-60, Mayor > 60 | Captura la no linealidad de la edad |

In [ ]:
# Creacion de nuevas variables sobre el dataset original
df_feat = df.copy()

df_feat['es_jubilado'] = (df_feat['trabajo'] == 'retired').astype(int)
df_feat['contactado_antes'] = (df_feat['dias_previos'] != 999).astype(int)
df_feat['contexto_favorable'] = (
    (df_feat['tasa_euribor'] < 2) & (df_feat['var_empleo'] < 0)
).astype(int)
df_feat['intensidad_campana'] = df_feat['campana'] / (df_feat['anterior'] + 1)

bins = [0, 30, 60, 100]
labels_edad = ['Joven (<=30)', 'Adulto (31-60)', 'Mayor (>60)']
df_feat['grupo_edad'] = pd.cut(df_feat['edad'], bins=bins, labels=labels_edad, right=True)

nuevas = ['es_jubilado', 'contactado_antes', 'contexto_favorable', 'intensidad_campana', 'grupo_edad']
for v in nuevas:
    print(f"{v}: dtype={df_feat[v].dtype}, valores unicos={df_feat[v].nunique()}")

In [ ]:
# Grafico: tasa de conversion por contexto macroeconomico
conv_contexto = df_feat.groupby('contexto_favorable')['deposito_plazo_num'].mean() * 100
etiquetas = ['Contexto Desfavorable', 'Contexto Favorable']
colores = ['#d63031', '#00b894']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(etiquetas, conv_contexto.values, color=colores, edgecolor='white', width=0.5)
for bar, val in zip(bars, conv_contexto.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.set_ylabel('Tasa de Conversion (%)')
ax.set_title('Tasa de Conversion por Contexto Macroeconomico (Variable creada: contexto_favorable)')
ax.set_ylim(0, conv_contexto.max() + 15)
sns.despine()
plt.tight_layout()
plt.savefig('graficos/ingenieria_conversion_por_contexto_macroeconomico.png', bbox_inches='tight')
plt.show()

In [ ]:
# Grafico: tasa de conversion por grupo de edad
conv_edad = df_feat.groupby('grupo_edad', observed=True)['deposito_plazo_num'].mean() * 100
colores_edad = ['#74b9ff', '#0984e3', '#6c5ce7']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(conv_edad.index.astype(str), conv_edad.values,
              color=colores_edad, edgecolor='white', width=0.55)
for bar, val in zip(bars, conv_edad.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.1f}%", ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.set_ylabel('Tasa de Conversion (%)')
ax.set_xlabel('Grupo Etario')
ax.set_title('Tasa de Conversion por Grupo de Edad (Variable creada: grupo_edad)')
ax.set_ylim(0, conv_edad.max() + 15)
sns.despine()
plt.tight_layout()
plt.savefig('graficos/ingenieria_conversion_por_grupo_edad.png', bbox_inches='tight')
plt.show()

In [ ]:
# Incorporar nuevas variables al dataset ml
df_ml['es_jubilado'] = df_feat['es_jubilado'].values
df_ml['contactado_antes'] = df_feat['contactado_antes'].values
df_ml['contexto_favorable'] = df_feat['contexto_favorable'].values
df_ml['intensidad_campana'] = df_feat['intensidad_campana'].values

grupo_dummies = pd.get_dummies(df_feat['grupo_edad'], prefix='grupo_edad', drop_first=True)
grupo_dummies.columns = [str(c) for c in grupo_dummies.columns]
df_ml = pd.concat([df_ml.reset_index(drop=True), grupo_dummies.reset_index(drop=True)], axis=1)

print(f"Shape final: {df_ml.shape}")
print(f"NaN totales: {df_ml.isnull().sum().sum()}")
vc = df_ml['deposito_plazo_num'].value_counts()
for val, cnt in vc.items():
    label = 'Si suscribe' if val == 1 else 'No suscribe'
    print(f"{label}: {cnt:,} ({cnt/len(df_ml)*100:.1f}%)")